<a href="https://colab.research.google.com/github/chikuu303/customer-churn-ml-project/blob/main/credit_risk_api_capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import os

print("Checking available project files...\n")

for filename in os.listdir("/content"):
    print(filename)

Checking available project files...

.config
sample_data


In [2]:

import pandas as pd
import numpy as np

# Create a reproducible example dataset
np.random.seed(42)
n = 1000

df = pd.DataFrame({
    "Age": np.random.randint(21, 66, n),
    "Income": np.random.randint(18000, 150000, n),
    "CreditScore": np.random.randint(300, 851, n),
    "Education": np.random.choice(
        ["High School", "Graduate", "Postgraduate"], n
    ),
    "EmploymentType": np.random.choice(
        ["Salaried", "Self-employed", "Unemployed"], n
    ),
    "LoanAmount": np.random.randint(5000, 60000, n),
    "MaritalStatus": np.random.choice(
        ["Single", "Married", "Divorced"], n
    )
})

# Create an example target using a simple risk rule
risk_score = (
    (df["CreditScore"] < 600).astype(int)
    + (df["Income"] < 35000).astype(int)
    + (df["LoanAmount"] > 35000).astype(int)
    + (df["EmploymentType"] == "Unemployed").astype(int)
)

df["Default"] = (
    (risk_score >= 2) |
    ((risk_score >= 1) & (np.random.random(n) < 0.25))
).astype(int)

print(df.head())
print("\nDataset shape:", df.shape)
print("\nTarget distribution:\n", df["Default"].value_counts())


from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
import joblib

# Separate input features and target
X = df.drop(columns=["Default"])
y = df["Default"]

numeric_features = [
    "Age", "Income", "CreditScore", "LoanAmount"
]

categorical_features = [
    "Education", "EmploymentType", "MaritalStatus"
]

# Preprocessing for numeric and categorical columns
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

# Combine preprocessing and model in one pipeline
credit_risk_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        class_weight="balanced"
    ))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

credit_risk_pipeline.fit(X_train, y_train)

print("Test accuracy:",
      round(credit_risk_pipeline.score(X_test, y_test), 4))

# Save the full pipeline
joblib.dump(
    credit_risk_pipeline,
    "credit_risk_pipeline.joblib"
)

print("Saved: credit_risk_pipeline.joblib")


from google.colab import files

files.download("credit_risk_pipeline.joblib")


   Age  Income  CreditScore     Education EmploymentType  LoanAmount  \
0   59  149149          809  Postgraduate     Unemployed        7537   
1   49  143501          845  Postgraduate     Unemployed       24226   
2   35   38611          526      Graduate       Salaried       20216   
3   63   32060          735   High School     Unemployed       23613   
4   28   61303          721   High School       Salaried        8009   

  MaritalStatus  Default  
0        Single        0  
1        Single        0  
2       Married        0  
3      Divorced        1  
4        Single        0  

Dataset shape: (1000, 8)

Target distribution:
 Default
1    578
0    422
Name: count, dtype: int64
Test accuracy: 0.85
Saved: credit_risk_pipeline.joblib


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:

%%writefile main.py
from fastapi import FastAPI
from pydantic import BaseModel, Field
import pandas as pd
import joblib

app = FastAPI(
    title="Credit Default Risk Prediction API",
    description="API for demonstrating credit default classification.",
    version="1.0.0"
)

model = joblib.load("credit_risk_pipeline.joblib")


class CreditApplication(BaseModel):
    Age: int = Field(ge=18, le=100)
    Income: float = Field(gt=0)
    CreditScore: int = Field(ge=300, le=850)
    Education: str
    EmploymentType: str
    LoanAmount: float = Field(gt=0)
    MaritalStatus: str


class PredictionResponse(BaseModel):
    prediction: int
    result: str
    default_probability: float


@app.get("/")
def home():
    return {
        "message": "Credit Default Risk API is running",
        "docs": "/docs"
    }


@app.get("/health")
def health():
    return {"status": "healthy"}


@app.post("/predict", response_model=PredictionResponse)
def predict(application: CreditApplication):
    input_df = pd.DataFrame([application.model_dump()])

    prediction = int(model.predict(input_df)[0])
    probabilities = model.predict_proba(input_df)[0]
    classes = list(model.classes_)
    default_probability = float(
        probabilities[classes.index(1)]
    )

    return {
        "prediction": prediction,
        "result": (
            "Default predicted" if prediction == 1
            else "No default predicted"
        ),
        "default_probability": round(default_probability, 4)
    }

Writing main.py


In [6]:

import os

print("main.py exists:", os.path.exists("main.py"))
print("Model exists:", os.path.exists("credit_risk_pipeline.joblib"))

main.py exists: True
Model exists: True


In [7]:

!pip install -q fastapi httpx joblib scikit-learn

In [8]:

from fastapi.testclient import TestClient
from main import app

client = TestClient(app)

# Test 1: Home endpoint
r1 = client.get("/")
print("Home:", r1.status_code, r1.json())

# Test 2: Health endpoint
r2 = client.get("/health")
print("Health:", r2.status_code, r2.json())

# Test 3: Prediction endpoint
customer = {
    "Age": 35,
    "Income": 55000,
    "CreditScore": 720,
    "Education": "Graduate",
    "EmploymentType": "Salaried",
    "LoanAmount": 15000,
    "MaritalStatus": "Married"
}

r3 = client.post("/predict", json=customer)
print("Prediction status:", r3.status_code)
print("Prediction output:", r3.json())

Home: 200 {'message': 'Credit Default Risk API is running', 'docs': '/docs'}
Health: 200 {'status': 'healthy'}
Prediction status: 200
Prediction output: {'prediction': 0, 'result': 'No default predicted', 'default_probability': 0.02}


In [9]:

%%writefile test_main.py
from fastapi.testclient import TestClient
from main import app

client = TestClient(app)

valid_customer = {
    "Age": 35,
    "Income": 55000,
    "CreditScore": 720,
    "Education": "Graduate",
    "EmploymentType": "Salaried",
    "LoanAmount": 15000,
    "MaritalStatus": "Married"
}


def test_home():
    response = client.get("/")
    assert response.status_code == 200


def test_health():
    response = client.get("/health")
    assert response.status_code == 200
    assert response.json()["status"] == "healthy"


def test_prediction_schema():
    response = client.post("/predict", json=valid_customer)
    assert response.status_code == 200

    result = response.json()
    assert result["prediction"] in [0, 1]
    assert result["result"] in [
        "Default predicted", "No default predicted"
    ]
    assert 0 <= result["default_probability"] <= 1


def test_missing_field():
    customer = valid_customer.copy()
    del customer["Income"]

    response = client.post("/predict", json=customer)
    assert response.status_code == 422


def test_invalid_age():
    customer = valid_customer.copy()
    customer["Age"] = 10

    response = client.post("/predict", json=customer)
    assert response.status_code == 422

Writing test_main.py


In [10]:

!pip install -q pytest
!pytest -v test_main.py

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: anyio-4.15.1, platformdirs-4.12.2, langsmith-0.14.2, typeguard-4.6.0
collected 5 items                                                              

test_main.py::test_home PASSED                                           [ 20%]
test_main.py::test_health PASSED                                         [ 40%]
test_main.py::test_prediction_schema PASSED                              [ 60%]
test_main.py::test_missing_field PASSED                                  [ 80%]
test_main.py::test_invalid_age PASSED                                    [100%]

============================== 5 passed in 1.73s ===============================


In [11]:

%%writefile requirements.txt
fastapi==0.115.12
uvicorn==0.34.0
pydantic==2.10.6
pandas==2.2.3
numpy==2.2.3
scikit-learn==1.6.1
joblib==1.4.2
httpx==0.28.1
pytest==8.3.5

Writing requirements.txt


In [12]:

%%writefile Dockerfile
FROM python:3.11-slim

WORKDIR /app

ENV PYTHONDONTWRITEBYTECODE=1
ENV PYTHONUNBUFFERED=1

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY main.py .
COPY credit_risk_pipeline.joblib .

EXPOSE 8000

CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]

Writing Dockerfile


In [13]:

%%writefile README.md
# Credit Default Risk Prediction API

## Project Overview
This project deploys a machine learning classifier through a REST API
built with FastAPI. It accepts customer and loan details and returns
a predicted default class and estimated default probability.

## Technology Stack
- Python
- Pandas and NumPy
- Scikit-learn
- FastAPI
- Uvicorn
- Docker
- Pytest

## Model
A Random Forest classifier is combined with a preprocessing pipeline.
Numerical features are imputed and scaled, while categorical features
are imputed and one-hot encoded.

The demonstration model was trained using synthetic data generated
with an illustrative risk rule. Its predictions are not suitable for
real lending decisions.

## Input Features
- Age
- Income
- CreditScore
- Education
- EmploymentType
- LoanAmount
- MaritalStatus

## API Endpoints
- GET / : API information
- GET /health : Health check
- POST /predict : Predict credit default risk

## Example Request
```json
{
  "Age": 35,
  "Income": 55000,
  "CreditScore": 720,
  "Education": "Graduate",
  "EmploymentType": "Salaried",
  "LoanAmount": 15000,
  "MaritalStatus": "Married"
}
```

## Example Response
```json
{
  "prediction": 0,
  "result": "No default predicted",
  "default_probability": 0.12
}
```

The response values above are illustrative, not guaranteed model output.

## Run Locally
Install dependencies:
pip install -r requirements.txt

Start the API:
uvicorn main:app --reload

Open http://127.0.0.1:8000/docs to explore the API.

## Run Tests
pytest -v test_main.py

## Run with Docker
Build the image:
docker build -t credit-risk-api .

Run the container:
docker run -p 8000:8000 credit-risk-api

## Limitations
The current model uses synthetic demonstration data. It must not be
used for real financial or lending decisions without suitable real
data, validation, fairness assessment, security, and monitoring.

Writing README.md


In [14]:

import os

for filename in [
    "main.py",
    "test_main.py",
    "requirements.txt",
    "Dockerfile",
    "README.md",
    "credit_risk_pipeline.joblib"
]:
    print(
        f"{filename}:",
        "Available" if os.path.exists(filename) else "MISSING"
    )

main.py: Available
test_main.py: Available
requirements.txt: Available
Dockerfile: Available
README.md: Available
credit_risk_pipeline.joblib: Available


In [15]:

import os
import zipfile

required_files = [
    "main.py",
    "test_main.py",
    "requirements.txt",
    "Dockerfile",
    "README.md",
    "credit_risk_pipeline.joblib"
]

missing_files = [
    name for name in required_files
    if not os.path.exists(name)
]

if missing_files:
    print("Missing files:", missing_files)
    print("Please create or restore these files before packaging.")
else:
    with zipfile.ZipFile(
        "credit-risk-api-capstone.zip", "w",
        zipfile.ZIP_DEFLATED
    ) as zipf:
        for name in required_files:
            zipf.write(name)

    print("Project ZIP created successfully!")
    print("File: credit-risk-api-capstone.zip")

Project ZIP created successfully!
File: credit-risk-api-capstone.zip


In [16]:

from google.colab import files

files.download("credit-risk-api-capstone.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>